**#05. Sentiment, Emotions, Topics, and Rating Prediction**

- **Instructor:** Jaeung Sim (University of Connecticut)
- **Course:** OPIM 5671 Data Mining and Time Series Forecasting
- **Last updated:** October 27, 2026

**Question:** Do additional text features improve predictions of review ratings?

Class sequence: separate Notebook 05, the full week 9 lecture, then this notebook.
Use Google Colab with the default CPU runtime. Run cells in order.

By the end, you will extract interpretable features, identify a few topics, compare
linear regression models fairly, and explain their performance on unseen reviews.
No GPU, transformer models, Gensim, or NLTK corpus downloads are required.


## 0. Colab setup
Only lightweight additions are installed. NumPy, pandas, matplotlib and scikit-learn
use Colab's existing versions; we print versions for reproducibility.
The dataset is downloaded once. Runtime depends on Colab availability and network.
We cap the working data at 3,000 reviews and fit each LDA candidate on at most
1,200 training reviews, 150 words per review and 1,500 vocabulary terms.
There are only three candidates and five iterations per fit.


In [ ]:
%pip -q install textblob==0.19.0 nrclex==4.1.0 kagglehub


In [ ]:
import re, time, random, importlib.metadata
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from textblob import TextBlob
from nrclex import NRCLex
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 25
MAX_REVIEWS = 3000
TOPIC_FIT_LIMIT = 1200
TOPIC_WORD_LIMIT = 150
TOPIC_COUNTS = (2, 3, 4)
EMOTIONS = ['anger','fear','anticipation','trust','surprise','sadness','joy','disgust']
EMOTION_COLS = [f'{e}_per100' for e in EMOTIONS]
for package in ['numpy','pandas','scikit-learn','textblob','nrclex']:
    print(package, importlib.metadata.version(package))


## 1. Data and prediction task
The source contains reviews of Disneyland branches and 1–5 star ratings.
Our target is `Rating`. Predictors come only from review text.
The goal is estimating a rating from an already-written review, not forecasting
future visits or estimating causal effects. Reviewer IDs and ratings are never predictors.


In [ ]:
# Automatic Kaggle download. If unavailable, upload DisneylandReviews.csv
# to Colab's Files pane and set LOCAL_CSV to its /content/... path.
LOCAL_CSV = None
if LOCAL_CSV:
    csv_path = Path(LOCAL_CSV)
else:
    import kagglehub
    data_path = Path(kagglehub.dataset_download('arushchillar/disneyland-reviews'))
    csv_files = sorted(data_path.rglob('*.csv'))
    if not csv_files:
        raise FileNotFoundError('Upload DisneylandReviews.csv and set LOCAL_CSV.')
    preferred = [p for p in csv_files if 'disneyland' in p.name.lower()]
    csv_path = (preferred or csv_files)[0]
raw = pd.read_csv(csv_path, encoding='latin-1')
required = {'Review_Text','Rating'}
assert required.issubset(raw.columns), f'Missing columns: {required - set(raw.columns)}'
print('Loaded:', csv_path.name, raw.shape)


In [ ]:
df = raw.dropna(subset=['Review_Text','Rating']).copy()
df['Review_Text'] = df['Review_Text'].astype(str).str.strip()
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df = df.loc[df['Review_Text'].ne('') & df['Rating'].isin([1,2,3,4,5])]
# Prevent identical review texts from appearing in different partitions.
df = df.drop_duplicates(subset=['Review_Text'])
df = df.sample(n=min(MAX_REVIEWS, len(df)), random_state=SEED).reset_index(drop=True)
df.index.name = 'row_id'
print('Working sample:', len(df))
display(df[['Review_Text','Rating']].head())


### A fair comparison
Reserve 20% for the final test. Split the remaining 80% into 80% training and 20%
validation: approximately 64% / 16% / 20% overall. All candidate models use the same rows.
Remove exact duplicate texts first. Near duplicates can still exist.
Use training data to fit transformations and models, validation to choose the
feature set, and test only once after decisions are fixed.


In [ ]:
# Reserve the test set first. Stratify to retain the rating imbalance.
dev, test = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df['Rating'])
train, valid = train_test_split(dev, test_size=0.20, random_state=SEED, stratify=dev['Rating'])
train, valid, test = train.copy(), valid.copy(), test.copy()
assert set(train.index).isdisjoint(valid.index)
assert set(dev.index).isdisjoint(test.index)
print({'train':len(train), 'validation':len(valid), 'test':len(test)})
train['Rating'].value_counts().sort_index().plot.bar(title='Training ratings')
plt.ylabel('Reviews'); plt.show()


## 2. Text preparation depends on the task
Keep original reviews for TextBlob so that “not good,” punctuation and intensifiers
remain available. For topics, lowercase, tokenize, remove stopwords and a small fixed
set of context words. We use simple lexical tokens rather than lemmatization to
keep setup short. This is a deliberate change from Notebook 05.


In [ ]:
def word_tokens(text):
    # Lightweight lexical matching. No NLTK corpus downloads are needed.
    return re.findall(r"[a-z]+(?:'[a-z]+)?", str(text).lower().replace('’', "'"))

TOPIC_STOPS = set(ENGLISH_STOP_WORDS) | {'disney','disneyland','park'}
def clean_for_topics(text):
    words = word_tokens(text)[:TOPIC_WORD_LIMIT]
    return ' '.join(w for w in words if len(w)>2 and w not in TOPIC_STOPS)

# Topic cleaning is deterministic. Learn the vocabulary later from training only.
# Sentiment receives the original text, preserving negation and punctuation.


In [ ]:
examples = ['The rides were good!', 'The rides were not good!', 'The rides were VERY good!']
display(pd.DataFrame({'Text':examples,
    'Polarity':[TextBlob(t).sentiment.polarity for t in examples],
    'Topic text':[clean_for_topics(t) for t in examples]}))


## 3. Sentiment and emotion features
TextBlob polarity ranges from −1 to +1. It is a lexicon/rule-based estimate,
not an observed rating. NRC associates words with eight emotions.
For each emotion, use **100 × matched token count / review token count**.
The denominator includes all lexical tokens, before topic stopword removal.
One word can match several emotions, so emotion rates need not sum to 100.
Retain `Review_Length` separately. NRC counts do not resolve negation, sarcasm
or context; these features describe lexical associations rather than measured feelings.


The two functions below match Assignment #2: `sentiment_features(frame)`
returns polarity; `emotion_features(frame)` returns length and eight emotion
rates. Both preserve row indices. `base_features(frame)` simply joins them.


In [ ]:
def sentiment_features(frame):
    # Score original text, preserving negation and punctuation.
    return pd.DataFrame({
        'TextBlob_Sentiment': frame['Review_Text'].map(
            lambda text: TextBlob(text).sentiment.polarity)
    }, index=frame.index)

emotion_reader = NRCLex()  # Load the bundled lexicon once.
def emotion_features(frame):
    records = []
    for text in frame['Review_Text']:
        tokens = word_tokens(text)
        emotion_reader.load_token_list(tokens)
        counts = dict(emotion_reader.raw_emotion_scores)
        record = {'Review_Length': len(tokens)}
        record.update({f'{e}_per100': 100*counts.get(e,0)/max(len(tokens),1)
                       for e in EMOTIONS})
        records.append(record)
    return pd.DataFrame(records, index=frame.index)

def base_features(frame):
    return sentiment_features(frame).join(emotion_features(frame))

started = time.perf_counter()
sent_train = sentiment_features(train)
sent_valid = sentiment_features(valid)
base_train = sent_train.join(emotion_features(train))
base_valid = sent_valid.join(emotion_features(valid))
print(f'Base features: {time.perf_counter()-started:.1f} seconds')
display(base_train.head())


In [ ]:
display(base_train.describe().T.round(3))
plt.scatter(base_train['TextBlob_Sentiment'],train['Rating'],alpha=0.12,s=12)
plt.xlabel('TextBlob polarity'); plt.ylabel('Observed rating'); plt.show()


**Discuss:** Why might two reviews with the same polarity have different ratings?
Why normalize emotion counts while retaining review length?


## 4. Required short topic modeling section
LDA represents a topic as a distribution over words and a review as a mixture
of topics. We use **word counts**, rather than TF–IDF, as the LDA input.
Topic numbers have no meaning until we inspect top words and example reviews.

Compare K = 2, 3, 4 using **UMass coherence** on the training topic sample.
It averages log co-occurrence scores among each topic's top eight words.
Higher (usually less negative) suggests more frequent co-occurrence; it does not
guarantee useful topics or better rating prediction. Inspect words before choosing K.
The helper below is provided; you do not need to memorize its implementation.


The count vectorizer uses `tokenizer=str.split` and `token_pattern=None`
to consume the already-cleaned tokens unchanged. Contractions stay intact;
the vectorizer does not turn “it's” or “don't” into new fragments after filtering.


In [ ]:
def top_words(model, vectorizer, n=8):
    terms = vectorizer.get_feature_names_out()
    return {f'Topic_{i+1}':list(terms[row.argsort()[-n:][::-1]])
            for i,row in enumerate(model.components_)}

def umass_coherence(model, counts, top_n=8):
    # UMass uses document co-occurrence among each topic's top words.
    # Higher (less negative) is better; this is NOT gensim's c_v score.
    present = (counts > 0).astype(np.int32).tocsc()
    scores = []
    for weights in model.components_:
        ids = weights.argsort()[-top_n:][::-1]
        pairs = []
        for m in range(1,len(ids)):
            for l in range(m):
                together = present[:,ids[m]].multiply(present[:,ids[l]]).sum()
                earlier = present[:,ids[l]].sum()
                pairs.append(np.log((float(together)+1)/max(float(earlier),1)))
        scores.append(np.mean(pairs))
    return float(np.mean(scores))

def new_lda(k):
    return LatentDirichletAllocation(n_components=k, learning_method='batch',
        max_iter=5, max_doc_update_iter=20, evaluate_every=-1,
        n_jobs=1, random_state=SEED)

def metrics(y, prediction):
    return {'MAE':mean_absolute_error(y,prediction),
            'MSE':mean_squared_error(y,prediction), 'R2':r2_score(y,prediction)}


In [ ]:
topic_fit = train.sample(n=min(TOPIC_FIT_LIMIT,len(train)), random_state=SEED)
vectorizer = CountVectorizer(preprocessor=clean_for_topics,
    max_features=1500, min_df=3, max_df=0.90,
    tokenizer=str.split, token_pattern=None, lowercase=False)
counts_fit = vectorizer.fit_transform(topic_fit['Review_Text'])
counts_train = vectorizer.transform(train['Review_Text'])
counts_valid = vectorizer.transform(valid['Review_Text'])

topic_models, topic_rows = {}, []
for k in TOPIC_COUNTS:
    started = time.perf_counter()
    lda = new_lda(k).fit(counts_fit)
    topic_models[k] = lda
    topic_rows.append({'K':k, 'UMass':umass_coherence(lda,counts_fit),
                       'Seconds':time.perf_counter()-started})
    print(f'K={k} completed in {topic_rows[-1]["Seconds"]:.1f}s')
topic_scores = pd.DataFrame(topic_rows)
display(topic_scores)
topic_scores.plot(x='K',y='UMass',marker='o',legend=False)
plt.xticks(TOPIC_COUNTS); plt.ylabel('UMass coherence (higher is better)'); plt.show()
for k, lda in topic_models.items():
    print(f'\nK={k}')
    display(pd.DataFrame(top_words(lda,vectorizer)))


### Choose K and interpret topic mixtures
Start with the highest coherence candidate. You may choose another candidate if
its topics are clearer, explaining the choice before final test evaluation.
Read at least two high-weight training reviews per topic. Propose tentative names
where the evidence supports them; otherwise explain why the topics are overlapping
or ambiguous. A small, fast LDA fit need not produce clearly separated themes.

The lecture's Waiting / Attractions / Service example is illustrative, not an
expected result. Do not force the observed topics into those labels. An evidence-based
explanation of ambiguity is a valid interpretation in Assignment #2.


In [ ]:
# Reproducible starting choice. After inspecting top words, an alternative K
# is allowed if you explain it BEFORE looking at test performance.
chosen_k = int(topic_scores.sort_values(['UMass','K'],ascending=[False,True]).iloc[0]['K'])
lda = topic_models[chosen_k]
print('Chosen K:',chosen_k)
display(pd.DataFrame(top_words(lda,vectorizer)))
topic_cols = [f'Topic_{i+1}' for i in range(chosen_k)]
topic_train = pd.DataFrame(lda.transform(counts_train),columns=topic_cols,index=train.index)
topic_valid = pd.DataFrame(lda.transform(counts_valid),columns=topic_cols,index=valid.index)
assert np.allclose(topic_train.sum(axis=1),1)
assert np.allclose(topic_valid.sum(axis=1),1)
display(topic_train.head())


In [ ]:
for topic in topic_cols:
    ids = topic_train[topic].nlargest(2).index
    print('\n',topic)
    display(train.loc[ids,['Review_Text','Rating']])


## 5. Feature sets and validation comparison
Compare a mean-rating baseline with:
1. Sentiment only.
2. Sentiment, eight emotion rates and review length.
3. The same features plus K−1 topic weights.

All K topic weights sum to one. With an intercept, omit the final topic weight
to avoid exact linear dependence. Do not use the mean topic weight; it is always 1/K.
Because model 2 adds emotions and length together, its improvement cannot be
attributed to emotion features alone.


In [ ]:
X_train = base_train.join(topic_train)
X_valid = base_valid.join(topic_valid)
feature_sets = {
    'Sentiment':['TextBlob_Sentiment'],
    'Sentiment + emotions + length':['TextBlob_Sentiment','Review_Length']+EMOTION_COLS,
    'Sentiment + emotions + length + topics':
        ['TextBlob_Sentiment','Review_Length']+EMOTION_COLS+topic_cols[:-1]
}
assert X_train.index.equals(train.index) and X_valid.index.equals(valid.index)
assert np.isfinite(X_train.to_numpy()).all()


### Evaluation
- **MAE:** average absolute error, in rating points. Primary selection metric.
- **MSE:** average squared error. More sensitive to large mistakes. Lower is better.
- **R²:** improvement over predicting the evaluation set's own mean. Higher is better;
  negative values are possible on unseen data.

The deployable mean baseline always uses the training mean, never the validation
or test mean. Select the lowest validation MAE, including the baseline. Exact ties
favor the earlier, simpler model. Do not select using training performance.


In [ ]:
models, comparison_rows = {}, []
baseline = DummyRegressor(strategy='mean').fit(np.zeros((len(train),1)),train['Rating'])
for name in ['Mean baseline']+list(feature_sets):
    if name == 'Mean baseline':
        model = baseline
        a, b = np.zeros((len(train),1)), np.zeros((len(valid),1))
    else:
        cols = feature_sets[name]
        a, b = X_train[cols], X_valid[cols]
        model = LinearRegression().fit(a,train['Rating'])
    models[name] = model
    for split, X, y in [('Training',a,train['Rating']),('Validation',b,valid['Rating'])]:
        comparison_rows.append({'Model':name,'Split':split,**metrics(y,model.predict(X))})
comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))
validation_results = comparison.query("Split == 'Validation'").sort_values('MAE',kind='stable')
selected_name = validation_results.iloc[0]['Model']
print('Selected by validation MAE:',selected_name)


**Discuss before continuing:** Which feature set wins validation MAE? Does adding
topics help? Is an improvement large enough to justify the extra work?
Record your choice now. Training fit alone cannot establish generalization.


## 6. Final held-out test
Keep the selected vocabulary, LDA and feature set fixed. Refit only the chosen
regression on the combined training + validation rows. Evaluate that model and a
development-mean baseline once on test. This simple workflow intentionally keeps
the topic representation frozen; a full refit would also have to regenerate all
topic features before refitting regression.


In [ ]:
# Run after freezing K and selected_name. Do not change them based on test results.
# Keep the training-fitted vocabulary and LDA fixed; refit regression on dev.
X_dev = pd.concat([X_train,X_valid]).sort_index()
y_dev = df.loc[X_dev.index,'Rating']
base_test = base_features(test)
counts_test = vectorizer.transform(test['Review_Text'])
topic_test = pd.DataFrame(lda.transform(counts_test),columns=topic_cols,index=test.index)
X_test = base_test.join(topic_test)
final_baseline = DummyRegressor(strategy='mean').fit(np.zeros((len(y_dev),1)),y_dev)
baseline_test_prediction = final_baseline.predict(np.zeros((len(test),1)))
if selected_name == 'Mean baseline':
    final_model = final_baseline
    final_prediction = baseline_test_prediction
else:
    selected_cols = feature_sets[selected_name]
    final_model = LinearRegression().fit(X_dev[selected_cols],y_dev)
    final_prediction = final_model.predict(X_test[selected_cols])
test_results = pd.DataFrame([
    {'Model':'Mean baseline',**metrics(test['Rating'],baseline_test_prediction)},
    {'Model':selected_name,**metrics(test['Rating'],final_prediction)}
]).drop_duplicates('Model')
display(test_results.round(4))
print('Predictions outside 1–5:', int(((final_prediction<1)|(final_prediction>5)).sum()))
# Evaluate raw regression predictions consistently. Do not round or clip them.
error_review = test[['Review_Text','Rating']].copy()
error_review['Prediction'] = final_prediction
error_review['Absolute_Error'] = abs(error_review['Rating']-final_prediction)
display(error_review.nlargest(5,'Absolute_Error'))


## 7. Interpretation and Assignment #2
Write a short conclusion answering:
- Did the selected model beat the mean baseline on the held-out test?
- What did sentiment, emotion rates and topics contribute on validation?
- What do two topics appear to describe, or why are they ambiguous? Use words and example reviews as evidence.
- What do the largest prediction errors reveal?
- What limits transfer to another park, language, or time period?

Ratings are bounded and ordinal; linear regression treats them as numeric and can
predict outside 1–5. We keep it as an interpretable teaching model and report raw
predictions consistently. This is a modest sample, not a claim of production accuracy.

The updated companion `OPIM5671_Assignment02_Week09_v2.ipynb` repeats this
workflow with completion and interpretation tasks.
Its required metrics are MAE, MSE and R². AIC/BIC and adjusted R² are not required.


## Optional extensions (discussion only)
Consider a sentiment + length model to isolate the incremental contribution of
emotions, different fixed stopwords, alternative seeds, or an ordinal model.
Do not rerun model selection after examining test results. Any new experiment
requires a new evaluation plan. No additional expensive execution is required.


## References
- [Disneyland reviews dataset](https://www.kaggle.com/datasets/arushchillar/disneyland-reviews)
- [TextBlob sentiment API](https://textblob.readthedocs.io/en/stable/api_reference.html)
- [NRCLex 4.1 documentation](https://pypi.org/project/NRCLex/4.1.0/)
- Mohammad, S. M. & Turney, P. D. (2013). Crowdsourcing a Word–Emotion Association Lexicon. Computational Intelligence, 29(3), 436–465. [NRC resource](https://saifmohammad.com/WebPages/NRC-Emotion-Lexicon.htm)
- Blei, D. M., Ng, A. Y. & Jordan, M. I. (2003). Latent Dirichlet Allocation. JMLR, 3, 993–1022. [Paper](https://www.jmlr.org/papers/v3/blei03a.html)
- [scikit-learn LDA](https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.LatentDirichletAllocation.html)
- [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- Mimno et al. (2011). Optimizing Semantic Coherence in Topic Models. [Paper](https://aclanthology.org/D11-1024/)
